# Detecção e Reconhecimento Facial

FIAP · MBA Data Science & Artificial Intelligence — Visão Computacional · Aula 04

## 1. Requerimentos

Todas as bibliotecas já estão instaladas no Google Colab.

* OpenCV >= 3.4.3
* Matplotlib >= 3.1.3
* Seaborn >= 0.0.10
* Numpy >= 1.18.1
* Mediapipe==0.10.9

### 1.2 Arquivos

Baixe o repositório do GitHub utilizando o comando abaixo. Em caso de atualização, utilize o comando para apagar o diretório antes.

In [ ]:
!rm -rf fiap-ml-visao-computacional/

In [ ]:
!git clone https://github.com/FIAPON/fiap-ml-visao-computacional.git

Vamos agora posicionar o diretório do repositório para a aula respectiva. Nesse caso envie o comando de mudança de diretório.

In [ ]:
%cd /content/fiap-ml-visao-computacional/aula-4-analise-facial

Importação das bibliotecas.

In [ ]:
!apt-get install -y libopencv-dev

A face é crucial na interação visual, sendo capaz de revelar informações não-verbais valiosas, como identificação, intenções e sentimentos, presentes em expressões faciais humanas. A análise de expressões faciais se torna uma área fascinante para quem estuda visão computacional, pois intersecta campos como reconhecimento de objetos, tratamento de imagens e identificação ou rastreio de pontos característicos (*landmarks*).

**Este notebook tem como foco a análise, detecção e reconhecimento facial**, um campo que tem ganhado destaque na área de inteligência artificial, uma vez que é possível obter uma riqueza de dados a partir de rostos por meio de algoritmos de visão computacional.


## Evolução dos Algoritmos de Detecção de Rosto

Ao longo do tempo, diversos algoritmos de detecção de rosto têm sido desenvolvidos, cada um com seus méritos e aplicações específicas. Iniciantes na área podem se sentir sobrecarregados pela diversidade de tutoriais disponíveis - uns focando em Haar Cascades, outros na biblioteca dlib e, mais recentemente, a ascensão do MediaPipe, que tem ganhado destaque a ponto de ofuscar alternativas tradicionais como o SSD (*Single Shot Multibox Detector*).

Para ilustrar essa evolução e ajudar na compreensão desse campo, a figura abaixo (retirada do artigo [What is Face Detection](https://learnopencv.com/what-is-face-detection-the-ultimate-guide/)) oferece um panorama dos algoritmos ao longo do tempo. **Mas como navegar por essa variedade sem se perder? Como tomar a melhor decisão para sua aplicação?**

### Escolhendo o modelo ideal

O modelo perfeito para detecção facial deve alinhar-se com os requisitos únicos do seu projeto. O autor do artigo mencionado sugere que a decisão deve ser pautada por três critérios fundamentais:

**Precisão de Detecção Superior:** Caso a prioridade seja capturar cada rosto com a maior precisão possível, os algoritmos DSFD ou RetinaFace-resnet50 emergem como as melhores escolhas. No entanto, é preciso estar ciente de que a alta complexidade de tais modelos resulta em uma velocidade de inferência reduzida, o que pode ser um empecilho para aplicações que exigem resposta em tempo real.

**Velocidade de Detecção Máxima:** Se o que importa é a rapidez da inferência, mesmo que isso signifique perder algumas detecções em cenários complexos, a solução de detecção facial do MediaPipe é a alternativa ideal.

**Melhor custo beneficio  entre velocidade e precisão:** Para aqueles que procuram um meio-termo, os modelos YuNet e RetinaFace-Mobilenetv1 apresentam-se como candidatos equilibrados. Eles oferecem uma velocidade de inferência rápida, adequada para aplicações em tempo real, sem sacrificar demasiadamente a precisão.

Selecionar o algoritmo correto é um passo decisivo que pode determinar o sucesso da sua aplicação.

In [ ]:
import numpy as np
import pandas as pd
import glob
from tqdm import tqdm
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import math
import time
import IPython

from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode


# Scikit learn
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn import preprocessing
from sklearn.svm import SVC

#Exibição na mesma tela do Jupyter
%matplotlib inline

import datetime
import dlib

from scipy.spatial import distance as dist

sns.set_style("whitegrid", {'axes.grid' : False})

Carregando um classificador pré-treinado de Haar.

Analise posteriormente outros classificadores disponíveis, dentre eles, classificador de pessoas, automóveis, gatos, sorriso, olhos, etc. neste repositório oficial do OpenCV https://github.com/opencv/opencv/tree/master/data/haarcascades.

## 2. Classificador de Viola-Jones (Haar Cascade)

Este classificador é especializado em identificar faces, também é conhecido como classificador Viola James e pode ser utilizado em outras aplicações. O paper original pode ser baixado [aqui](https://www.cs.cmu.edu/~efros/courses/LBMV07/Papers/viola-cvpr-01.pdf).

Devido a característica deste tipo de classificador, sua identificação é extramamente rápida, com identificação < 0,02s, aplicações em sistemas em tempo real, especialmente câmeras de vigilância.

### 2.1 Classificador de Faces

A aplicação mais utilizada do classificador em cascata de Viola-Jones (ou classificador de Haar) é para encontrar faces em imagens ou vídeos, especialmente por sua identificação ser bem rápida.

Este tipo de classificador serve para separar a região de interesse de uma determinada área, para posteriormente, aplicar outros classificadores que poderão, por exemplo, classificar de quem é o rosto, uma vez que a região foi separado da imagem original.

In [ ]:
# Carregando classifcador
classificador_face = cv2.CascadeClassifier('classificadores/haarcascade_frontalface_default.xml')

imagem = cv2.imread('/content/fiap-ml-visao-computacional/aula-4-analise-facial/imagens/purchase.jpg')
#imagem = cv2.imread('imagens/college.jpg')

imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)
imagem_gray = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.figure(figsize=(20,10))
plt.imshow(imagem)
plt.axis("off")

Retornará a região de interesse da face identificada como tupla, armazenando as coordenadas superiores esquerda e inferior  direita.

Se retornar vazio é por que não há faces identificadas.
Os valores padrão são configurações inciais recomendadas

`objects = cascade.detectMultiScale(image, scaleFactor, minNeighbors, flags, minSize, maxSize)`

Parâmetros

`image (obrigatório)`:
* Imagem de entrada (normalmente em escala de cinza).
* Pode ser uma matriz do NumPy carregada pelo cv2.imread() ou cv2.cvtColor().

`scaleFactor (opcional, padrão: 1.1)`:
* Define quanto o tamanho da imagem é reduzido em cada escala.
* Um valor menor que 1.1 pode aumentar a precisão, mas tornará a detecção mais lenta.
* Um valor maior acelera o processamento, mas pode perder detalhes.

`minNeighbors (opcional, padrão: 3–5 recomendado)`:
* Número de vizinhos que um candidato a detecção precisa ter para ser aceito.
* Valores maiores reduzem falsos positivos, mas podem descartar objetos reais.
* Valores menores podem resultar em mais detecções, mas com maior probabilidade de erros.

`flags (opcional, padrão: 0)`:
* Parâmetro legado, geralmente não precisa ser alterado.
* Pode ser usado para configurações como cv2.CASCADE_SCALE_IMAGE, mas não é necessário na maioria dos casos.

`minSize (opcional, padrão: Nenhum, ajustado automaticamente)`:
* Tamanho mínimo do objeto a ser detectado (largura, altura).
* Evita detectar objetos muito pequenos que podem ser ruídos.

`maxSize (opcional, padrão: Nenhum, ajustado automaticamente)`:
* Tamanho máximo do objeto a ser detectado (largura, altura).
* Pode ser útil para evitar detectar objetos muito grandes e irrelevantes.


In [ ]:
faces = classificador_face.detectMultiScale(imagem_gray, 1.3, 5)

# Lista de faces. Caso não seja identificada será retornado None (nulo)
if faces is None:
     cv2.putText(imagem, "Rosto ausente", (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,0), 6)

# Desenhando retângulos nos rostos identificados
for i, (x,y,w,h) in enumerate(faces):
    roi = imagem[y:y+h, x:x+w]
    roi = cv2.cvtColor(roi, cv2.COLOR_RGB2BGR)

    cv2.imwrite("face_" + str(i) + ".png", roi)
    cv2.rectangle(imagem, (x,y), (x+w,y+h), (255,255,0), 2)

plt.figure(figsize=(20,10))
plt.imshow(imagem)
plt.axis('off')
plt.title("Faces identificadas")

### 2.2 Aplicação com Webcam (Tempo Real)

Para acessar a webcam do PC no google colab é precisar recorrer ao JavaScript. As funções abaixo mostra um exemplo de como fazer essa conexão

In [ ]:
import cv2
import numpy as np
import time
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode

def setup_camera():
    js_code = """
    var old = document.getElementById('camera-container');
    if (old) old.remove();

    window._stopRequested = false;

    var div = document.createElement('div');
    div.id = 'camera-container';
    div.style.cssText = 'font-family: monospace; padding: 10px;';
    document.body.appendChild(div);

    div.innerHTML = `
        <h3 style="color:#4CAF50;">Detector de Faces — Viola-Jones (Haar Cascade)</h3>
        <video id="webcam" autoplay playsinline style="display:none;"></video>
        <canvas id="canvas" style="display:none;"></canvas>
        <div style="position:relative; display:inline-block;">
            <img id="output" style="display:block; margin:auto; border:3px solid #4CAF50; border-radius:8px; max-width:640px;">
            <div id="face-counter" style="
                position:absolute; top:8px; left:8px;
                background:rgba(0,0,0,0.6); color:#4CAF50;
                padding:4px 10px; border-radius:4px; font-size:14px; font-weight:bold;">
                Faces: 0
            </div>
            <div id="fps-counter" style="
                position:absolute; top:8px; right:8px;
                background:rgba(0,0,0,0.6); color:#FFC107;
                padding:4px 10px; border-radius:4px; font-size:14px;">
                FPS: --
            </div>
        </div>

        <div style="margin-top:10px; display:flex; align-items:center; gap:12px;">
            <div id="status-bar" style="color:#888; font-size:12px;">
                Inicializando câmera...
            </div>

            <!-- BOTÃO DE PARAR -->
            <button id="stop-btn" onclick="
                window._stopRequested = true;
                this.disabled = true;
                this.innerText = 'Desativando ...';
                this.style.background = '#555';
            " style="
                background: #e53935;
                color: white;
                border: none;
                padding: 6px 16px;
                border-radius: 6px;
                font-size: 13px;
                font-weight: bold;
                cursor: pointer;
                display: flex;
                align-items: center;
                gap: 6px;
            ">
                &#9632; Desativar
            </button>
        </div>
    `;

    window._cameraStream = null;

    async function setupCamera() {
        const video = document.getElementById('webcam');
        const stream = await navigator.mediaDevices.getUserMedia({
            video: { width: 640, height: 480 },
            audio: false,
        });
        window._cameraStream = stream;
        video.srcObject = stream;
        return new Promise((resolve) => {
            video.onloadedmetadata = () => {
                document.getElementById('status-bar').innerHTML = 'Detecção de Faces ativa';
                resolve();
            };
        });
    }

    async function captureFrame() {
        const video = document.getElementById('webcam');
        const canvas = document.getElementById('canvas');
        const ctx = canvas.getContext('2d');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        return canvas.toDataURL('image/jpeg', 0.85);
    }

    window.captureFrame = captureFrame;
    setupCamera();
    """
    display(Javascript(js_code))
    time.sleep(3)


def stop_camera():
    js_stop = """
    if (window._cameraStream) {
        window._cameraStream.getTracks().forEach(track => track.stop());
        window._cameraStream = null;
    }
    var statusBar = document.getElementById('status-bar');
    if (statusBar) {
        statusBar.innerHTML = 'Câmera desligada.';
        statusBar.style.color = '#f44336';
    }
    var btn = document.getElementById('stop-btn');
    if (btn) {
        btn.innerText = 'Desativado';
        btn.style.background = '#388E3C';
    }
    """
    eval_js(js_stop)
    print("Câmera desligada.")


def should_stop():
    """
    Verifica se o botão foi clicado consultando a flag JS.
    Retorna True se _stopRequested === true no navegador.
    """
    try:
        return bool(eval_js('window._stopRequested === true'))
    except:
        return False


def clear_output_display():
    display(Javascript('google.colab.output.clearAll();'))
    time.sleep(1)


def capture_frame():
    js_result = eval_js('captureFrame()')
    if not js_result:
        return None
    return b64decode(js_result.split(',')[1])


def update_display(frame, n_faces=0, fps=0.0):
    _, buffer = cv2.imencode('.jpg', frame)
    frame_b64 = b64encode(buffer).decode('utf-8')
    eval_js(f"""
    document.getElementById('output').src = 'data:image/jpeg;base64,{frame_b64}';
    document.getElementById('face-counter').innerText = 'Faces: {n_faces}';
    document.getElementById('fps-counter').innerText = 'FPS: {fps:.1f}';
    """)


face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def detect_faces(frame):
    if frame is None:
        return None, 0
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    n_faces = len(faces)
    for i, (x, y, w, h) in enumerate(faces):
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, f"Face {i+1}", (x, y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
    return frame, n_faces


def face_detection_loop():
    clear_output_display()
    setup_camera()

    print("Detecção facial iniciada.")
    print("Clique em [ ■ Desativar Camera ] na interface ou interrompa a célula.\n")

    frame_count = 0
    start_time = time.time()

    try:
        while True:
            if should_stop():
                print("\nDesativado pelo botão da interface.")
                break

            frame_binary = capture_frame()
            if frame_binary is None:
                time.sleep(0.5)
                continue

            frame = cv2.imdecode(np.frombuffer(frame_binary, dtype=np.uint8), -1)
            processed_frame, n_faces = detect_faces(frame)

            frame_count += 1
            elapsed = time.time() - start_time
            fps = frame_count / elapsed if elapsed > 0 else 0.0

            update_display(processed_frame, n_faces=n_faces, fps=fps)
            time.sleep(0.05)

    except KeyboardInterrupt:
        print("\nLoop interrompido pelo usuário.")
    except Exception as e:
        print(f"\nErro inesperado: {e}")
    finally:
        stop_camera()


face_detection_loop()


## 3. LBPH — Local Binary Pattern Histogram

> ⚠️ **Atenção:** É necessário reiniciar o kernel antes de executar esta seção.

In [ ]:
!pip uninstall opencv_contrib_python
!pip install opencv_contrib_python

In [ ]:

!unzip faces.zip

In [ ]:
%cd /content/fiap-ml-visao-computacional/aula-4-analise-facial

In [ ]:
# Importando as bibliotecas necessárias
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow


# Definindo o caminho para as imagens
faces_folder_path = 'faces/Train'

# Listando as imagens
image_files = [f for f in os.listdir(faces_folder_path) if f.endswith('.jpg')]

# Visualizando algumas imagens do dataset
def plot_images(images, titles, rows, cols):
    fig, axes = plt.subplots(rows, cols, figsize=(15, 10))
    axes = axes.ravel()
    for i in range(rows * cols):
        image = cv2.imread(os.path.join(faces_folder_path, images[i]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        axes[i].imshow(image)
        axes[i].set_title(titles[i])
        axes[i].axis('off')
    plt.subplots_adjust(hspace=0.5)
    plt.show()

# Plotando 6 imagens para visualização inicial
plot_images(image_files[:6], image_files[:6], 2, 3)


In [ ]:
# Carregando as imagens e preparando os dados para treinamento
faces = []
labels = []
label_dict = {}
label_counter = 0

# Definindo o caminho para o diretório com imagens
faces_folder_path = 'faces/Train'
image_files = os.listdir(faces_folder_path)

for i, file_name in enumerate(image_files):
    image_path = os.path.join(faces_folder_path, file_name)
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is not None:
        faces.append(image)
        # Usando um label único para cada imagem (pode ser modificado para agrupar por pessoa)
        labels.append(label_counter)
        label_dict[label_counter] = file_name
        label_counter += 1

faces = np.array(faces)
labels = np.array(labels)

In [ ]:
# Treinando o reconhecedor
lbph_recognizer = cv2.face.LBPHFaceRecognizer_create()
lbph_recognizer.train(faces, labels)

In [ ]:
# Importando as bibliotecas necessárias
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import local_binary_pattern

# Função para calcular e plotar os padrões LBP de uma imagem
def plot_lbp(image_path, radius=1, n_points=8):

    # Carregando a imagem em escala de cinza
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        print("Erro: Imagem não encontrada ou inválida.")
        return

    # Calculando o padrão LBP da imagem
    lbp = local_binary_pattern(image, n_points, radius, method='uniform')

    # Calculando o histograma do LBP
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))

    # Normalizando o histograma
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    # Plotando a imagem original, a imagem LBP e o histograma
    plt.figure(figsize=(18, 6))

    # Imagem original
    plt.subplot(1, 3, 1)
    plt.imshow(image, cmap='gray')
    plt.title('Imagem Original')
    plt.axis('off')

    # Imagem LBP
    plt.subplot(1, 3, 2)
    plt.imshow(lbp, cmap='gray')
    plt.title('Padrão Local Binary Pattern (LBP)')
    plt.axis('off')

    # Histograma do LBP
    plt.subplot(1, 3, 3)
    plt.bar(range(len(hist)), hist, width=0.6, color='b')
    plt.title('Histograma do LBP')
    plt.xlabel('Padrões LBP')
    plt.ylabel('Frequência Normalizada')

    plt.tight_layout()
    plt.show()

# Chamando a função para visualizar o LBP da imagem de teste
plot_lbp(test_image_path)


In [ ]:
def recognize_face(image_path):
    # Carregando imagem de teste
    test_img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if test_img is None:
        print("Erro: Imagem de teste não encontrada ou inválida.")
        return

    # Executando o reconhecimento
    label, confidence = lbph_recognizer.predict(test_img)

    # Encontrando o nome do label
    recognized_name = label_dict.get(label, "Desconhecido")
    recognized_image_path = os.path.join(faces_folder_path, label_dict[label])
    recognized_img = cv2.imread(recognized_image_path, cv2.IMREAD_GRAYSCALE)

    # Concatenando as imagens lado a lado
    combined_image = np.hstack((test_img, recognized_img))

    # Exibindo os resultados
    print(f"Face reconhecida: {recognized_name} com confiança de {confidence}")
    print("Imagem de teste e imagem mais similar encontrada:")
    cv2_imshow(combined_image)


In [ ]:
recognize_face('faces/Teste/image_0047.jpg')

## 4. Detecção com DNN do OpenCV (SSD)

O OpenCV oferece ferramentas robustas para a detecção de rostos através do seu módulo de Redes Neurais Profundas (DNN - *Deep Neural Network*). Este módulo incorpora a capacidade de execução de passagens diretas (inferência) utilizando redes neurais pré-treinadas provenientes de diversos frameworks, tais como Caffe, TensorFlow, Torch e Darknet.

Desde a versão 3.1 do OpenCV, o módulo DNN tem se tornado um componente valioso, uma vez que permite a realização de inferências sem a necessidade de treinamento da rede. A partir da versão 3.3, o módulo foi incorporado ao repositório principal do OpenCV, passando por otimizações e acelerações significativas.

### *Single Shot Multibox Detector* - SSD

O detector de rosto disponibilizado pelo OpenCV é construído sobre o framework SSD (*Single Shot MultiBox Detector*) utilizando uma arquitetura de rede ResNet-10. A capacidade deste detector reside na sua habilidade de identificar e localizar rostos em imagens, fazendo uso de modelos pré-treinados que simplificam a implementação e integração em diversas aplicações.

O OpenCV disponibiliza dois modelos específicos para a detecção de rosto:

1. **Detector de Rosto (FP16)**: Esta versão utiliza uma representação *float-point** de 16 bits da implementação original em Caffe.
2. **Detector de Rosto (UINT8)**: Trata-se de uma versão quantizada de 8 bits utilizando TensorFlow.

Para a utilização de ambos os modelos, são necessários dois conjuntos de arquivos: **o arquivo de modelo, contendo os pesos das camadas da rede, e o arquivo de configuração, que define a arquitetura da rede neural.**

#### Modelo TensorFlow
- **opencv_face_detector_uint8.pb**: Contém os pesos das camadas da rede.
- **opencv_face_detector.pbtxt**: Define a arquitetura do modelo.

### Aplicação Prática do Módulo DNN


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

imagem = cv2.imread("/content/fiap-ml-visao-computacional/aula-4-analise-facial/imagens/piscada.jpg")
imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(20,10))
plt.imshow(imagem)
plt.axis('off')

### 4.1 Leitura do Modelo DNN

A função readNetFromCaffe é utilizada para carregar um modelo de rede neural pré-treinado armazenado no formato do framework Caffe. Esta função é frequentemente utilizada na preparação de modelos para detecção e classificação em tarefas de visão computacional. Ela lê os arquivos de descrição da arquitetura da rede e os pesos aprendidos durante o treinamento e cria uma instância da rede neural para inferência.

#### <font style="color:rgb(0, 0, 128)">Sintaxe da Função</font>

```python
cv2.dnn.readNetFromCaffe(
    prototxt: str,
    caffeModel: str
) -> cv2.dnn_Net
```

**Parâmetros**<br>

- **prototxt** (*str*): Caminho para o arquivo .prototxt que contém a descrição textual da arquitetura da rede neural.
- **caffeModel** (*str*): Caminho para o arquivo .caffemodel que contém os pesos da rede neural treinada.

**Retorno**<br>

- **cv2.dnn_Net**: Retorna uma instância da rede neural para inferência.


In [ ]:
model = cv2.dnn.readNet('modelos/opencv_face_detector.pbtxt', 'modelos/opencv_face_detector_uint8.pb')

### 4.2 Conversão de Imagem para Blob

A função blobFromImage é usada para criar um blob 4-dimensional a partir de uma imagem. Opcionalmente, redimensiona e corta a imagem a partir do centro, subtrai valores médios, escala valores pelo fator de escala e troca os canais Azul e Vermelho.

#### <font style="color:rgb(0, 0, 128)">Sintaxe da Função</font>

```python
cv2.dnn.blobFromImage(
    image: np.ndarray,
    scalefactor: Optional[float] = 1.0,
    size: Optional[Tuple[int, int]] = (0, 0),
    mean: Optional[Tuple[float, float, float]] = (0.0, 0.0, 0.0),
    swapRB: Optional[bool] = True,
    crop: Optional[bool] = False,
    ddepth: Optional[int] = cv2.CV_32F
) -> np.ndarray
```

**Parâmetros**<br>

- **image** (*np.ndarray*): Imagem de entrada (com 1, 3 ou 4 canais).
- **scalefactor** (*float, opcional*): Multiplicador para os valores da imagem.
- **size** (*Tuple[int, int], opcional*): Tamanho espacial para a imagem de saída.
- **mean** (*Tuple[float, float, float], opcional*): Escalar com valores médios que são subtraídos dos canais.
- **swapRB** (*bool, opcional*): Indica se a troca dos canais primeiro e último em uma imagem de 3 canais é necessária.
- **crop** (*bool, opcional*): Indica se a imagem será cortada após o redimensionamento.
- **ddepth** (*int, opcional*): Profundidade do blob de saída. Escolha entre CV_32F ou CV_8U.

**Retorno**<br>

- **np.ndarray**: Retorna a imagem de saída do mesmo tamanho e profundidade que a imagem de entrada.

In [ ]:
# Cria um blob a partir da imagem
img_copy = imagem.copy()

blob = cv2.dnn.blobFromImage(img_copy, 1.0, (300, 300), [104, 117, 123], True, False)

### 4.3 Configurando o Valor de Entrada

A função setInput define um novo valor de entrada para a rede neural.

#### <font style="color:rgb(0, 0, 128)">Sintaxe da Função</font>

```python
cv2.dnn_Net.setInput(
    blob: np.ndarray,
    name: Optional[str] = None,
    scalefactor: Optional[float] = None,
    mean: Optional[Tuple[float, float, float]] = None
) -> None
```

**Parâmetros**<br>

- **blob** (*np.ndarray*): Novo blob.
- **name** (*str, opcional*): Nome da camada de entrada.
- **scalefactor** (*float, opcional*): Fator de normalização opcional.
- **mean** (*Tuple[float, float, float], opcional*): Valores de subtração de média opcionais.

**Retorno**<br>

- **None**

In [ ]:
# Define o blob como entrada para o modelo
model.setInput(blob)

### 4.4 Inferência e Visualização de Detecções

A função forward executa uma passagem para frente para calcular a saída de uma camada com o nome outputName. Retorna o blob para a primeira saída da camada especificada.

#### <font style="color:rgb(0, 0, 128)">Sintaxe da Função</font>

```python
cv2.dnn_Net.forward(
    outputName: Optional[str] = None
) -> np.ndarray
```

**Parâmetros**<br>

- **outputName** (*str, opcional*): Nome da camada para a qual a saída é necessária.

**Retorno**<br>

- **np.ndarray**: Retorna o blob para a primeira saída da camada especificada.

In [ ]:
detections = model.forward()

Agora, já com as detecções, primeiramente são extraídas as dimensões da imagem de entrada `img_copy`, especificamente a altura (`h`) e a largura (`w`). Isso é feito para posteriormente ajustar as coordenadas dos retângulos delimitadores das detecções em relação ao tamanho real da imagem. Em seguida, o código entra em um laço para iterar sobre todas as detecções disponíveis, que estão armazenadas na matriz `detections`.

Dentro *loop*, para cada detecção, se obtém o valor de confiança associado, que indica quão confiante o modelo está de que a região detectada contém um rosto. Um *threshold* é definido, e somente as detecções que superam esse limiar são consideradas. Para essas detecções, o código calcula as coordenadas do retângulo delimitador ajustadas ao tamanho da imagem, converte essas coordenadas para inteiros e, em seguida, desenha o retângulo e exibe o valor de confiança na imagem `img_copy`.


In [ ]:
# Obtém as dimensões da imagem
(h, w) = img_copy.shape[:2]

# Itera sobre as detecções
for i in range(detections.shape[2]):
    # Obtém a confiança da detecção
    confidence = detections[0, 0, i, 2]

    # Define um limiar de confiança para considerar uma detecção válida
    confidence_threshold = 0.999

    # Verifica se a confiança é maior que o limiar
    if confidence > confidence_threshold:
        # Obtém as coordenadas do retângulo delimitador
        box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])

        # Converte as coordenadas para inteiros
        (startX, startY, endX, endY) = box.astype("int")

        # Desenha o retângulo delimitador na imagem
        cv2.rectangle(img_copy, (startX, startY), (endX, endY), (0, 255, 0), 2)

        # Adiciona a confiança da detecção na imagem
        text = "{:.2f}%".format(confidence * 100)
        cv2.putText(img_copy, text, (startX, startY - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 2)

plt.figure(figsize=(20,10))
plt.imshow(img_copy)
plt.axis('off')

## 5. Identificação de Marcos Faciais com Dlib

O toolkit [Dlib](http://dlib.net) fornece uma série de bibliotecas e ferramentas para utilização em problemas de visão computacional. Para o uso de identificação de marcos faciais vamos utilizar sua biblioteca responsável por identificar faces de uma imagem e posterior identificação dos marcos faciais.

O Dlib vem instalado por padrão na plataforma Google Colab, não é necesário baixar.

Os modelos podem ser baixados neste endereço http://dlib.net/files/. Os que serão utilizados estão já incluídos na pasta ```modelos```.

Iniciamos o processo de identificação configurando o modelo treinado de 68 pontos ```shape_predictor_68_face_landmarks.dat```. Após isso precisamos identificar a face e para cada face identificada retornar a lista de pontos, na variável ```marcos_faciais```.

In [ ]:
# Caminho do arquivo do modelo pretreinado para a detecção de 68 pontos de referência faciais
classificador_68_path = "modelos/shape_predictor_68_face_landmarks.dat"

# Caminho do arquivo do modelo pretreinado para a detecção de 5 pontos de referência faciais
classificador_5_path = "modelos/shape_predictor_5_face_landmarks.dat"

# Carrega o modelo do Dlib para detectar 68 pontos de referência no rosto
classificador_dlib_68 = dlib.shape_predictor(classificador_68_path)

# Carrega o modelo do Dlib para detectar 5 pontos de referência no rosto
classificador_dlib_5 = dlib.shape_predictor(classificador_5_path)

# Inicializa o detector de rostos do Dlib baseado em HOG + SVM
detector_face_dlib = dlib.get_frontal_face_detector()



In [ ]:
def obter_marcos(imagem, marcos_68_pontos=True):
    """
    Detecta marcos faciais (landmarks) em uma imagem usando o modelo do Dlib.

    Parâmetros:
    - imagem: imagem de entrada onde os rostos serão detectados.
    - marcos_68_pontos (bool): define se deve usar o modelo de 68 pontos (True) ou 5 pontos (False).

    Retorna:
    - Lista de matrizes contendo os marcos faciais de cada rosto detectado.
    - Retorna None se nenhum rosto for encontrado.
    """

    # Seleciona o classificador adequado com base na quantidade de pontos desejada
    classificador_dlib = classificador_dlib_68
    if not marcos_68_pontos:
        classificador_dlib = classificador_dlib_5

    # Detecta rostos na imagem usando o detector do Dlib (HOG + SVM)
    faces = detector_face_dlib(imagem, 1)  # O "1" indica que a imagem será redimensionada uma vez para melhorar a detecção

    # Se nenhum rosto for detectado, retorna None
    if len(faces) == 0:
        return None

    marcos_faciais = []  # Lista para armazenar os marcos faciais detectados

    # Itera sobre os rostos detectados
    for face in faces:
        shape = classificador_dlib(imagem, face)  # Obtém os marcos faciais para o rosto atual

        pontos = [] # Re-initialize pontos as a list for each face
        # Coleta as coordenadas dos pontos faciais
        num_points = 68 if marcos_68_pontos else 5
        for i in range(0, num_points):
            pontos.append([shape.part(i).x, shape.part(i).y])

        # Converte a lista de pontos para uma matriz NumPy
        pontos_matrix = np.matrix(pontos) # Convert to matrix after collecting all points for the face

        # Adiciona os marcos faciais à lista
        marcos_faciais.append(pontos_matrix) # Append the matrix to the list

    return marcos_faciais  # Retorna a lista com os marcos faciais dos rostos detectados

Com os pontos mapeados, o próximo passo é criar uma função para anotar em uma nova imagem.

In [ ]:
def anotar_marcos(imagem, marcos_faciais):
    """
    Desenha pontos dos marcos faciais na imagem.

    Parâmetros:
    - imagem: a imagem onde os marcos faciais serão desenhados.
    - marcos_faciais: lista de matrizes contendo os pontos dos marcos faciais.

    Retorna:
    - A imagem com os marcos faciais anotados.
    """

    # Verifica se há marcos faciais detectados
    if marcos_faciais is None:
        print("Não foi identificado nenhum marco facial.")
        return imagem  # Retorna a imagem original sem modificações

    # Percorre cada conjunto de marcos faciais detectados
    for marco_facial in marcos_faciais:
        for idx, ponto in enumerate(marco_facial):
            # Obtém a posição do ponto (x, y)
            centro = (ponto[0, 0], ponto[0, 1])

            # Desenha um círculo amarelo no ponto correspondente
            cv2.circle(imagem, centro, 3, (0, 255, 255), -1)  # -1 preenche o círculo

    return imagem  # Retorna a imagem com os marcos desenhados


In [ ]:
imagem = cv2.imread("imagens/doctor.jpg")

# Obtém os marcos faciais da imagem
marcos_faciais = obter_marcos(imagem)
# Desenha os marcos faciais na imagem
imagem_marcos = anotar_marcos(imagem, marcos_faciais)

imagem_marcos = cv2.cvtColor(imagem_marcos, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(30,20))
plt.axis('off')
plt.imshow(imagem_marcos)

Aplicando a detecção e anotação utilizando uma câmera.

In [ ]:
import IPython.display as display
from google.colab import output
import cv2
import numpy as np
import base64
import io
import os

def take_photo(filename='imagens/foto.jpg', quality=0.8):
    """
    Ativa a câmera e captura uma foto quando o usuário clica no botão "Capturar Foto".

    Parâmetros:
    - filename (str): Caminho onde a imagem será salva.
    - quality (float): Qualidade da imagem entre 0 e 1.

    Retorna:
    - filename (str): Caminho do arquivo salvo.
    """

    js = """
    async function capturePhoto(quality) {
        return new Promise((resolve, reject) => {
            const div = document.createElement('div');
            document.body.appendChild(div);

            const video = document.createElement('video');
            video.style.width = '100%';
            video.style.maxWidth = '640px';
            video.style.height = 'auto';
            div.appendChild(video);

            navigator.mediaDevices.getUserMedia({ video: true }).then(stream => {
                video.srcObject = stream;
                video.play();

                const button = document.createElement('button');
                button.innerText = '📷 Capturar Foto';
                button.style.display = 'block';
                button.style.marginTop = '10px';
                button.style.padding = '10px';
                button.style.fontSize = '16px';
                button.style.cursor = 'pointer';
                button.style.backgroundColor = 'red';
                button.style.color = 'white';
                div.appendChild(button);

                button.onclick = () => {
                    const canvas = document.createElement('canvas');
                    canvas.width = video.videoWidth;
                    canvas.height = video.videoHeight;
                    canvas.getContext('2d').drawImage(video, 0, 0);
                    stream.getTracks().forEach(track => track.stop());  // Para a captura de vídeo
                    div.remove();  // Remove elementos do DOM

                    let dataUrl = canvas.toDataURL('image/jpeg', quality);
                    resolve(dataUrl);  // Retorna a imagem capturada
                };
            }).catch(err => {
                div.remove();
                reject(err);
            });
        });
    }
    """

    try:
        # Garante que o diretório de destino existe
        os.makedirs(os.path.dirname(filename), exist_ok=True)

        display.display(display.Javascript(js))
        data_url = output.eval_js('capturePhoto({})'.format(quality))

        if not data_url or not isinstance(data_url, str) or ',' not in data_url:
            raise ValueError("Erro: Nenhuma imagem foi capturada. Tente novamente.")

        # Remove o prefixo 'data:image/jpeg;base64,' da string Base64
        encoded_data = data_url.split(',')[1]

        try:
            # Decodifica os dados Base64 em bytes com tratamento de padding
            # Adiciona padding se necessário
            padding = len(encoded_data) % 4
            if padding:
                encoded_data += '=' * (4 - padding)

            image_data = base64.b64decode(encoded_data)
        except Exception as e:
            raise ValueError(f"Erro na decodificação Base64: {str(e)}")

        # Verifica se a conversão gerou bytes válidos
        if not image_data or len(image_data) == 0:
            raise ValueError("Erro: Dados da imagem vazios após decodificação Base64.")

        # Converte os bytes para um array NumPy
        image_array = np.frombuffer(image_data, dtype=np.uint8)

        # Verifica se o array não está vazio antes de tentar decodificar com OpenCV
        if image_array.size == 0:
            raise ValueError("Erro: O array de imagem está vazio. Tente novamente.")

        # Decodifica a imagem no formato OpenCV
        image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)

        # Verifica se a imagem foi corretamente carregada
        if image is None:
            raise ValueError("Erro: OpenCV falhou ao decodificar a imagem.")

        # Salva a imagem corretamente
        cv2.imwrite(filename, image)

        print(f"📷 Imagem salva em: {filename}")
        return filename
    except Exception as e:
        raise RuntimeError(f"Erro ao acessar a câmera: {str(e)}")


# Execução do código
from IPython.display import Image as ColabImage
try:
    filename = take_photo("imagens/foto.jpg")

    # Exibe a imagem salva
    display.display(ColabImage(filename))
except Exception as err:
    print("Erro:", str(err))

In [ ]:
imagem = cv2.imread("imagens/foto.jpg")

marcos_faciais = obter_marcos(imagem)
imagem_marcos = anotar_marcos(imagem, marcos_faciais)

imagem_marcos = cv2.cvtColor(imagem_marcos, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(30,20))
plt.axis('off')
plt.imshow(imagem_marcos)
plt.title("Marcos faciais da câmera")

### 5.1 Análise de Marcos Faciais

Os 68 pontos estão divididos nos seguintes coponentes do rosto humano:

* Face completa ```1 ao 68```
* Sombrancelha direita ```17 ao 21```
* Sombracelha esquerda ```22 ao 26```
* Olho direito ```36 ao 41```
* Olho esquerdo ```42 ao 47```
* Nariz ```27 ao 34```
* Lábio ```48 ao 60```
* Lábio exterior ``47 a 60``
* Lábio interior ``59 a 67``  
* Mandíbula ```1 ao 16```

Os pontos que o DLib retorna são zero based, portanto o primeiro ponto começa no 0.

In [ ]:
FACE_COMPLETA = list(range(0, 68))
SOMBRANCELHA_DIREITA = list(range(17, 22))
SOMBRANCELHA_ESQUERDA = list(range(22, 27))
OLHO_DIREITO = list(range(36, 42))
OLHO_ESQUERDO = list(range(42, 48))
NARIZ = list(range(27, 35))
LABIO = list(range(48, 61))
LABIO_EXTERIOR = list(range(48, 61))
LABIO_INTERIOR = list(range(60, 68))
MANDIBULA = list(range(0, 17))

Função para criar polígonos a partir dos pontos de identificação. Estes polígonos podem oferecer ferramentas como cálculo de área para identificação melhor das estruturas faciais, como piscadas, movimentação da boca, dentre outros.

In [ ]:
def anotar_marcos_faciais_componente(imagem, marcos_faciais):
    """
    Desenha contornos ao redor de componentes faciais (lábios e olhos) usando os marcos faciais detectados.

    Parâmetros:
    - imagem: A imagem onde os contornos serão desenhados.
    - marcos_faciais: Lista contendo os marcos faciais detectados.

    Retorna:
    - A imagem com os contornos desenhados.
    """

    # Verifica se há marcos faciais detectados
    if marcos_faciais is None:
        print("Não foi identificado nenhum marco facial.")
        return imagem  # Retorna a imagem original sem alterações

    # Percorre cada conjunto de marcos faciais (um para cada rosto detectado)
    for marco_facial in marcos_faciais:
        for d in marco_facial:  # Parece redundante, pode ser removido

            # Obtém os pontos convexos ao redor dos lábios e desenha o contorno em verde
            pontos = cv2.convexHull(marco_facial[LABIO])
            cv2.drawContours(imagem, [pontos], 0, (0, 255, 0), 2)

            # Obtém os pontos convexos ao redor do olho esquerdo e desenha o contorno em verde
            pontos = cv2.convexHull(marco_facial[OLHO_ESQUERDO])
            cv2.drawContours(imagem, [pontos], 0, (0, 255, 0), 2)

            # Obtém os pontos convexos ao redor do olho direito e desenha o contorno em verde
            pontos = cv2.convexHull(marco_facial[OLHO_DIREITO])
            cv2.drawContours(imagem, [pontos], 0, (0, 255, 0), 2)

    return imagem  # Retorna a imagem com os contornos desenhados


In [ ]:
imagem = cv2.imread("imagens/cervero.jpeg")

marcos_faciais = obter_marcos(imagem)
imagem_marcos = anotar_marcos_faciais_componente(imagem, marcos_faciais)

imagem_marcos = cv2.cvtColor(imagem_marcos, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(30,20))
plt.axis('off')
plt.imshow(imagem_marcos)

In [ ]:
imagem = cv2.imread("imagens/bruce.jpg")

marcos_faciais = obter_marcos(imagem)
imagem_marcos = anotar_marcos_faciais_componente(imagem, marcos_faciais)

imagem_marcos = cv2.cvtColor(imagem_marcos, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(30,20))
plt.axis('off')
plt.imshow(imagem_marcos)

### 5.2 EAR — Eye Aspect Ratio

Para lidar com situações onde o rosto capturado pode se aproximar ou se afastar da câmera utilizamos um cálculo de aspecto de razão. Esse cálculo divide duas medidas e sua razão é utilizada como medida para análises.

Uma vez que temos os pontos em volta de cada componente do rosto humano é possível realizar essas medidas de aspecto de razão para analisar a abertura dos olhos, por exemplo.

Nesse sentido, calculamos a medida de aspecto razão dos olhos ou EAR. Essa medida foi discutida neste [paper de Soukupová and Čech](http://vision.fe.uni-lj.si/cvww2016/proceedings/papers/05.pdf).

In [ ]:
def ear(pontos_olho):
    """
    Calcula a Razão de Aspecto do Olho (EAR - Eye Aspect Ratio), que é usada para detectar fechamento dos olhos.

    Parâmetros:
    - pontos_olho: Lista de pontos que representam os marcos faciais do olho.

    Retorna:
    - medida_ear: Um valor que representa a abertura do olho. Valores menores indicam que o olho está fechado.
    """

    # Converter os pontos do olho em vetores 1-D para facilitar o cálculo
    ponto1 = np.array([pontos_olho[1][0, 0], pontos_olho[1][0, 1]])  # Canto interno superior
    ponto2 = np.array([pontos_olho[2][0, 0], pontos_olho[2][0, 1]])  # Superior do meio
    ponto3 = np.array([pontos_olho[3][0, 0], pontos_olho[3][0, 1]])  # Inferior do meio
    ponto4 = np.array([pontos_olho[4][0, 0], pontos_olho[4][0, 1]])  # Canto interno inferior
    ponto5 = np.array([pontos_olho[5][0, 0], pontos_olho[5][0, 1]])  # Canto externo inferior

    # Calcular as distâncias verticais entre os pontos do olho
    a = dist.euclidean(ponto2, ponto5)  # Distância entre a parte superior e inferior do meio do olho
    b = dist.euclidean(ponto3, ponto4)  # Outra distância vertical do olho
    c = dist.euclidean(ponto1, ponto4)  # Distância horizontal entre os cantos do olho

    # Calcular a Razão de Aspecto do Olho (EAR)
    medida_ear = (a + b) / (2.0 * c)  # Fórmula usada para detectar fechamento ocular

    return medida_ear  # Quanto menor o valor, mais fechado está o olho

def obter_pontos_olho(marcos_faciais, olho_indices):
    """
    Obtém os pontos dos olhos a partir dos marcos faciais detectados.

    Parâmetros:
    - marcos_faciais: Matriz contendo todos os marcos faciais detectados.
    - olho_indices: Lista de índices dos pontos específicos do olho.

    Retorna:
    - pontos_olho: Lista contendo os pontos do olho desejado.
    """

    pontos_olho = []  # Lista para armazenar os pontos do olho

    # Percorre os índices dos marcos faciais e adiciona os pontos correspondentes à lista
    for idx in olho_indices:
        pontos_olho.append(marcos_faciais[0][idx])

    return pontos_olho  # Retorna os pontos do olho selecionado

Vamos inspecionar a abertura dos olhos da imagem analisada. Note que como existe apenas uma única pessoa (ou rosto) na imagem,  utilizaremos o índice 0 somente. Se houvessem mais pessoas, poderíamos analisar cada uma delas iterando sobre o índice.

In [ ]:
pontos_olho_direito = obter_pontos_olho(marcos_faciais, OLHO_DIREITO)
ear_olho_direito = ear(pontos_olho_direito)
print("EAR do olho direito " + str(ear_olho_direito))

pontos_olho_esquerdo = obter_pontos_olho(marcos_faciais, OLHO_ESQUERDO)
ear_olho_esquerdo = ear(pontos_olho_esquerdo)
print("EAR do olho esquerdo " + str(ear_olho_esquerdo))

In [ ]:
imagem = cv2.imread("imagens/person-closed-yes.jpg")

marcos_faciais = obter_marcos(imagem)
imagem_marcos = anotar_marcos_faciais_componente(imagem, marcos_faciais)

imagem_marcos = cv2.cvtColor(imagem_marcos, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(30,20))
plt.axis('off')
plt.imshow(imagem_marcos)

In [ ]:
pontos_olho_direito = obter_pontos_olho(marcos_faciais, OLHO_DIREITO)
ear_olho_direito = ear(pontos_olho_direito)
print("EAR do olho direito " + str(ear_olho_direito))

pontos_olho_esquerdo = obter_pontos_olho(marcos_faciais, OLHO_ESQUERDO)
ear_olho_esquerdo = ear(pontos_olho_esquerdo)
print("EAR do olho esquerdo " + str(ear_olho_esquerdo))

In [ ]:
imagem = cv2.imread("imagens/amazed.jpg")

marcos_faciais = obter_marcos(imagem)
imagem_marcos = anotar_marcos_faciais_componente(imagem, marcos_faciais)

imagem_marcos = cv2.cvtColor(imagem_marcos, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(30,20))
plt.axis('off')
plt.imshow(imagem_marcos)

In [ ]:
pontos_olho_direito = obter_pontos_olho(marcos_faciais, OLHO_DIREITO)
ear_olho_direito = ear(pontos_olho_direito)
print("EAR do olho direito " + str(ear_olho_direito))

pontos_olho_esquerdo = obter_pontos_olho(marcos_faciais, OLHO_ESQUERDO)
ear_olho_esquerdo = ear(pontos_olho_esquerdo)
print("EAR do olho esquerdo " + str(ear_olho_esquerdo))

Aplicar a análise de face e marcos faciais utilizando a câmera.

In [ ]:
imagem = cv2.imread("imagens/foto.jpg")

marcos_faciais = obter_marcos(imagem)
imagem_marcos = anotar_marcos_faciais_componente(imagem, marcos_faciais)

imagem_marcos = cv2.cvtColor(imagem_marcos, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(30,20))
plt.axis('off')
plt.imshow(imagem_marcos)
plt.title("Marcos faciais com 68 pontos")

In [ ]:
pontos_olho_direito = obter_pontos_olho(marcos_faciais, OLHO_DIREITO)
ear_olho_direito = ear(pontos_olho_direito)
print("EAR do olho direito " + str(ear_olho_direito))

pontos_olho_esquerdo = obter_pontos_olho(marcos_faciais, OLHO_ESQUERDO)
ear_olho_esquerdo = ear(pontos_olho_esquerdo)
print("EAR do olho esquerdo " + str(ear_olho_esquerdo))

### 5.3 Alinhamento de Faces

Uma técnica de pré-processamento de imagens contendo faces humanas é o alinhamento dos olhos. Esta técnica torna as regiões entre as difernetes imagens em uma mesma localização espacial, o que torna o aprendizado de reconhecimento de faces mais efetivo.

In [ ]:
def extrair_olho(shape, eye_indices):
    """
    Extrai os pontos faciais correspondentes a um olho.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.
    - eye_indices: Lista de índices dos pontos faciais que representam o olho.

    Retorna:
    - Lista de pontos do olho.
    """
    points = map(lambda i: shape.part(i), eye_indices)
    return list(points)

def extrair_olho_centro(shape, eye_indices):
    """
    Calcula o ponto central de um olho baseado nos seus marcos faciais.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.
    - eye_indices: Lista de índices dos pontos faciais do olho.

    Retorna:
    - Tupla (x, y) representando o centro do olho.
    """
    points = extrair_olho(shape, eye_indices)
    xs = map(lambda p: p.x, points)  # Extrai coordenadas x dos pontos do olho
    ys = map(lambda p: p.y, points)  # Extrai coordenadas y dos pontos do olho
    return sum(xs) // 6, sum(ys) // 6  # Retorna a média das coordenadas como centro

def extrair_olho_centro_esquerdo(shape):
    """
    Obtém o centro do olho esquerdo.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.

    Retorna:
    - Tupla (x, y) representando o centro do olho esquerdo.
    """
    return extrair_olho_centro(shape, OLHO_ESQUERDO)

def extrair_olho_centro_esquerdo(shape):
    """
    Obtém o centro do olho esquerdo.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.

    Retorna:
    - Tupla (x, y) representando o centro do olho esquerdo.
    """
    return extrair_olho_centro(shape, OLHO_ESQUERDO)


def extrair_olho_centro_direito(shape):
    """
    Obtém o centro do olho direito.

    Parâmetros:
    - shape: Objeto dlib contendo os marcos faciais.

    Retorna:
    - Tupla (x, y) representando o centro do olho direito.
    """
    return extrair_olho_centro(shape, OLHO_DIREITO)

def angulo_entre_pontos(p1, p2):
    """
    Calcula o ângulo de inclinação entre dois pontos.

    Parâmetros:
    - p1, p2: Tuplas (x, y) representando dois pontos no espaço.

    Retorna:
    - O ângulo em graus entre os dois pontos.
    """
    x1, y1 = p1
    x2, y2 = p2
    tan = (y2 - y1) / (x2 - x1)  # Calcula a tangente do ângulo
    return np.degrees(np.arctan(tan))  # Converte o ângulo para graus

def get_rotation_matrix(p1, p2):
    """
    Calcula a matriz de rotação para alinhar a face.

    Parâmetros:
    - p1: Tupla (x, y) do olho esquerdo.
    - p2: Tupla (x, y) do olho direito.

    Retorna:
    - Matriz de rotação do OpenCV para alinhamento.
    """
    angle = angulo_entre_pontos(p1, p2)  # Calcula o ângulo de inclinação da face
    x1, y1 = p1
    x2, y2 = p2
    xc = (x1 + x2) // 2  # Calcula o ponto médio entre os olhos
    yc = (y1 + y2) // 2
    M = cv2.getRotationMatrix2D((xc, yc), angle, 1)  # Obtém a matriz de rotação
    return M

def align_face(image_path, tamanho=None, return_face=False):
    """
    Alinha a face em uma imagem com base na posição dos olhos.

    Parâmetros:
    - image_path: Caminho da imagem de entrada.
    - tamanho: Tupla opcional (largura, altura) para redimensionamento.
    - return_face: Se True, retorna a face alinhada em vez de exibir.

    Retorna:
    - Se `return_face` for True, retorna a imagem da face alinhada.
    - Caso contrário, exibe a imagem original e alinhada.
    """
    imagem = cv2.imread(image_path)
    imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)

    height, width = imagem.shape[:2]
    dets = detector_face_dlib(imagem, 1)

    if len(dets) == 0:
        return None

    for i, det in enumerate(dets):

        x = det.left()
        y = det.top()
        w = det.right() - x
        h = det.bottom() - y

        # Garante que as coordenadas não fiquem fora dos limites da imagem
        if x < 0: x = 0
        if y < 0: y = 0

        roi = imagem[y:y+h, x:x+w]  # Recorta a região do rosto detectado

        # Obtém os marcos faciais do rosto detectado
        shape = classificador_dlib_68(imagem, det)
        left_eye = extrair_olho_centro_esquerdo(shape)  # Centro do olho esquerdo
        right_eye = extrair_olho_centro_direito(shape)  # Centro do olho direito

        M = get_rotation_matrix(left_eye, right_eye)  # Matriz de rotação

        # Define tamanho personalizado se especificado
        if tamanho:
            width = tamanho[0]
            height = tamanho[1]

        # Aplica a rotação na imagem
        rotated = cv2.warpAffine(imagem, M, (width, height))

        # Recorta novamente a região do rosto após a rotação
        cropped = rotated[y:y+h, x:x+w]

    if return_face:
        return cropped

    plt.subplot(1,2,1)
    plt.title("Original")
    plt.imshow(roi)

    plt.subplot(1,2,2)
    plt.title("Alinhado")
    plt.imshow(cropped)

    plt.show()

In [ ]:
align_face("imagens/Aaron_Eckhart_0001.jpg")

In [ ]:
align_face("imagens/Aaron_Guiel_0001.jpg")

In [ ]:
align_face("imagens/Aaron_Pena_0001.jpg")

### 5.4 Estimação de Pose da Cabeça (Yaw, Pitch, Roll)


A rotação de rosto (head pose estimation) é uma técnica de visão computacional usada para determinar a orientação da cabeça de uma pessoa em relação à câmera. Ela é expressa em três ângulos principais:

- Pitch: Inclinação para cima ou para baixo (movimento de "sim").
- Yaw: Inclinação para os lados, esquerda ou direita (movimento de "não").
- Roll: Inclinação rotacional, como inclinar a cabeça para os ombros.

Esses ângulos são amplamente utilizados em aplicações como reconhecimento facial, realidade aumentada e monitoramento de atenção.

Passos para Estimar a Rotação de Rosto

1. **Detecção de Face**: Utilizamos o detector de face do Dlib para identificar a região do rosto na imagem ou vídeo.
2. **Detecção de Landmarks**: Uma vez que a face é detectada, o Dlib identifica 68 pontos de landmarks faciais.
3. **Cálculo dos Ângulos**: Com esses pontos, usamos algoritmos de projeção 3D para calcular os ângulos de pitch, yaw e roll, que indicam a orientação da cabeça.

In [ ]:
import cv2
import dlib
import numpy as np

# Carregar o detector de faces e o predictor de landmarks
detector_face = dlib.get_frontal_face_detector()
predictor_landmarks = dlib.shape_predictor("modelos/shape_predictor_68_face_landmarks.dat")

# Função para definir pontos de referência 3D da face
MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0),             # Nariz
    (0.0, -330.0, -65.0),        # Ponta do queixo
    (-225.0, 170.0, -135.0),     # Canto do olho esquerdo
    (225.0, 170.0, -135.0),      # Canto do olho direito
    (-150.0, -150.0, -125.0),    # Canto da boca esquerdo
    (150.0, -150.0, -125.0)      # Canto da boca direito
])



In [ ]:
# Função para definir a matriz da câmera
def get_camera_matrix(size):
    """
    Esta função calcula a matriz intrínseca da câmera, que é usada para projetar
    pontos 3D no espaço 2D da imagem.

    Parâmetros:
    - size: Tupla contendo as dimensões (altura, largura) da imagem.

    A matriz da câmera (ou matriz intrínseca) inclui:
    - Focal length (comprimento focal), que é uma medida da distância entre a lente e o sensor da câmera.
      Aqui, estamos usando a largura da imagem como uma aproximação para o comprimento focal.
    - Center (centro da imagem), que é o ponto onde o eixo óptico da câmera encontra o sensor.
      É representado pelo ponto central da imagem.

    Retorno:
    - Uma matriz 3x3 de valores `double` que representa a matriz da câmera.

    Exemplo de estrutura da matriz:
        [[focal_length, 0, center_x],
         [0, focal_length, center_y],
         [0, 0, 1]]

    Onde:
    - focal_length é aproximadamente a largura da imagem.
    - center_x e center_y representam o ponto central da imagem.
    """

    focal_length = size[1]  # Usamos a largura da imagem como aproximação para o comprimento focal
    center = (size[1] / 2, size[0] / 2)  # Calcula o centro da imagem (ponto médio)
    return np.array([[focal_length, 0, center[0]],
                     [0, focal_length, center[1]],
                     [0, 0, 1]], dtype="double")

In [ ]:
# Função para calcular a orientação da cabeça e projetar os eixos
def estimate_head_pose_and_draw_axes(image, landmarks):
    """
    Esta função calcula a orientação da cabeça (pitch, yaw e roll) com base em pontos de referência faciais (landmarks)
    e desenha os eixos X, Y e Z sobre a imagem para ilustrar a orientação.

    Parâmetros:
    - image: A imagem onde os eixos serão desenhados.
    - landmarks: Os pontos de referência faciais (landmarks) detectados no rosto, usados para calcular a pose.

    Passos:
    1. **Pontos de Referência 2D**: Define pontos específicos na face (nariz, queixo, olhos e boca) para serem usados
       no cálculo da orientação.
    2. **Matriz da Câmera**: Calcula a matriz intrínseca da câmera usando as dimensões da imagem.
    3. **Cálculo de Rotação e Translação**: Utiliza o algoritmo `solvePnP` do OpenCV para estimar a rotação e a
       translação da cabeça em relação à câmera.
    4. **Cálculo dos Ângulos Pitch, Yaw e Roll**: Converte o vetor de rotação em uma matriz de rotação e, em seguida,
       calcula os ângulos de rotação em graus:
       - **Pitch**: Inclinação para cima ou para baixo.
       - **Yaw**: Inclinação para os lados (esquerda/direita).
       - **Roll**: Rotação como se estivesse inclinando a cabeça para os ombros.
    5. **Projeção dos Eixos 3D para 2D**: Define os eixos X, Y e Z e projeta-os para o espaço 2D da imagem, de forma
       que possam ser desenhados.
    6. **Desenho dos Eixos**: Desenha os eixos X, Y e Z na imagem para mostrar visualmente a orientação da cabeça.
       - Vermelho para o eixo X (esquerda/direita).
       - Verde para o eixo Y (cima/baixo).
       - Azul para o eixo Z (frente/trás).
    7. **Exibição dos Ângulos**: Adiciona texto na imagem com os valores de pitch, yaw e roll.

    Retorno:
    - A imagem modificada, com os eixos e os valores de pitch, yaw e roll desenhados sobre ela.

    Exemplo de uso:
    - Esta função é útil para aplicações em visão computacional onde a orientação da cabeça deve ser monitorada,
      como realidade aumentada, análise de atenção e reconhecimento facial.

    """

    # Pontos de referência 2D
    image_points = np.array([
        (landmarks.part(30).x, landmarks.part(30).y),  # Nariz
        (landmarks.part(8).x, landmarks.part(8).y),    # Queixo
        (landmarks.part(36).x, landmarks.part(36).y),  # Olho esquerdo
        (landmarks.part(45).x, landmarks.part(45).y),  # Olho direito
        (landmarks.part(48).x, landmarks.part(48).y),  # Boca esquerda
        (landmarks.part(54).x, landmarks.part(54).y)   # Boca direita
    ], dtype="double")

    # Matriz da câmera e distorção zero
    camera_matrix = get_camera_matrix(image.shape)
    dist_coeffs = np.zeros((4, 1))

    # Calcular rotação e translação
    _, rotation_vector, translation_vector = cv2.solvePnP(MODEL_POINTS, image_points, camera_matrix, dist_coeffs)

    # Convertendo o vetor de rotação em uma matriz de rotação
    rotation_matrix, _ = cv2.Rodrigues(rotation_vector)

    # Calculando pitch, yaw e roll a partir da matriz de rotação
    sy = np.sqrt(rotation_matrix[0, 0] ** 2 + rotation_matrix[1, 0] ** 2)
    pitch = np.arctan2(rotation_matrix[2, 1], rotation_matrix[2, 2])
    yaw = np.arctan2(-rotation_matrix[2, 0], sy)
    roll = np.arctan2(rotation_matrix[1, 0], rotation_matrix[0, 0])

    # Converte para graus
    pitch, yaw, roll = [np.degrees(angle) for angle in (pitch, yaw, roll)]

    # Definir os pontos dos eixos 3D (X: vermelho, Y: verde, Z: azul)
    axis_length = 100
    axis_points_3d = np.float32([
        [axis_length, 0, 0],      # Eixo X
        [0, axis_length, 0],      # Eixo Y
        [0, 0, axis_length]       # Eixo Z
    ])

    # Projeção dos pontos dos eixos para o espaço 2D da imagem
    nose_end_point2D, _ = cv2.projectPoints(axis_points_3d, rotation_vector, translation_vector, camera_matrix, dist_coeffs)
    nose_point = (int(image_points[0][0]), int(image_points[0][1]))

    # Desenhar os eixos com conversão explícita para inteiros
    for i, color in enumerate([(0, 0, 255), (0, 255, 0), (255, 0, 0)]):  # Vermelho para X, Verde para Y, Azul para Z
        end_point = tuple(map(int, nose_end_point2D[i].ravel()))
        cv2.line(image, nose_point, end_point, color, 3)

    # Adicionar texto dos valores de pitch, yaw e roll
    cv2.putText(image, f'Pitch: {pitch:.2f}', (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.putText(image, f'Yaw: {yaw:.2f}', (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.putText(image, f'Roll: {roll:.2f}', (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    return image


In [ ]:
# Caminho do vídeo de entrada e do vídeo de saída
input_video_path = 'videos/video_cabeca_rotacao.mp4'  # Substitua pelo caminho do seu vídeo
output_video_path = 'processed_video.mp4'

# Abrir o vídeo de entrada
cap = cv2.VideoCapture(input_video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Definir o codec e criar o objeto de escrita de vídeo
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Processar cada quadro
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector_face(gray)

    # Processa apenas a primeira face detectada
    for face in faces:
        landmarks = predictor_landmarks(gray, face)
        frame = estimate_head_pose_and_draw_axes(frame, landmarks)
        break  # Processar apenas a primeira face para reduzir carga de processamento

    # Escrever o quadro processado no vídeo de saída
    out.write(frame)

# Libere os objetos
cap.release()
out.release()
print("Processamento de vídeo concluído e salvo como 'processed_video.mp4'")


In [ ]:
!apt-get install -y ffmpeg
!ffmpeg -i processed_video.mp4 -vcodec libx264 -acodec aac processed_video_converted.mp4

In [ ]:
from IPython.display import HTML
from base64 import b64encode

# Função para exibir o vídeo no notebook
def show_video(video_path, width=640):
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""<video width={width} controls><source src="{data_url}" type="video/mp4"></video>""")

# Exibir o vídeo convertido
show_video("processed_video.mp4")


## 6. MediaPipe FaceMesh

*MediaPipe* é uma plataforma de código aberto para a criação de soluções de percepção situacional. Fornecida pelo Google, ela oferece uma estrutura para construir aplicativos e soluções que necessitam de processamento de fluxos de dados multimídia, incluindo, mas não se limitando a, vídeo, áudio e dados de sensores. *MediaPipe* foi criado com a missão de acelerar o ciclo de desenvolvimento de aplicações multimídia em várias plataformas.

Para a tarefa de *Pose Estimation*, nós iremos utilizar o *MediaPipe* para:

1. **Detectar e desenhar *pose landmarks***: *MediaPipe* nos permitirá identificar os principais pontos do corpo, ou *landmarks*, e desenhá-los em nossas imagens e vídeos para uma visualização mais intuitiva.
2. **Desenhar *landmark connections***: Além de detectar *landmarks* individuais, o *MediaPipe* também permite desenhar conexões entre estes, facilitando a compreensão da postura geral.
3. **Obter o pixel de coordenada da *landmark***: Com o *MediaPipe*, nós poderemos extrair as coordenadas dos pixels correspondentes a cada *landmark*, permitindo análises mais detalhadas.

### 4.3.1 - Pose Landmark Model - BlazePose GHUM 3D

O modelo de *landmark* para estimativa de pose que utilizaremos é o *BlazePose GHUM 3D*, também oferecido pelo Google através do *MediaPipe*. Este modelo fornece uma representação completa de 3D do corpo humano, incluindo 33 *landmarks* que cobrem todo o corpo.<br><br>

<img src="https://sigmoidal.ai/wp-content/uploads/2023/07/68747470733a2f2f6d65646961706970652e6465762f696d616765732f6d6f62696c652f706f73655f747261636b696e675f66756c6c5f626f64795f6c616e646d61726b732e706e67.png" width=500>

<br>O modelo [*BlazePose GHUM 3D*](https://github.com/google/mediapipe/blob/master/docs/solutions/pose.md) foi treinado em um grande conjunto de dados de poses e movimentos humanos, permitindo-lhe prever com precisão os *landmarks* corporais mesmo em diferentes poses e orientações. Isso o torna uma ferramenta poderosa para nossas análises de desempenho esportivo. Veja a lista completa dos *landmarks*:

* 0 - Nariz
* 1 - Olho esquerdo (interno)
* 2 - Olho esquerdo
* 3 - Olho esquerdo (externo)
* 4 - Olho direito (interno)
* 5 - Olho direito
* 6 - Olho direito (externo)
* 7 - Orelha esquerda
* 8 - Orelha direita
* 9 - Boca (lado esquerdo)
* 10 - Boca (lado direito)
* 11 - Ombro esquerdo
* 12 - Ombro direito
* 13 - Cotovelo esquerdo
* 14 - Cotovelo direito
* 15 - Pulso esquerdo
* 16 - Pulso direito
* 17 - Dedo mínimo esquerdo
* 18 - Dedo mínimo direito
* 19 - Dedo indicador esquerdo
* 20 - Dedo indicador direito
* 21 - Polegar esquerdo
* 22 - Polegar direito
* 23 - Quadril esquerdo
* 24 - Quadril direito
* 25 - Joelho esquerdo
* 26 - Joelho direito
* 27 - Tornozelo esquerdo
* 28 - Tornozelo direito
* 29 - Calcanhar esquerdo
* 30 - Calcanhar direito
* 31 - Dedão do pé esquerdo
* 32 - Dedão do pé direito

Cada um desses *landmarks* representa um ponto chave no corpo humano, permitindo-nos analisar a pose e o movimento do corpo de forma abrangente e precisa.


In [ ]:
!pip install mediapipe==0.10.13

In [ ]:
# Importando bibliotecas necessárias
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
from moviepy.editor import VideoFileClip


### 6.1 Detecção em Imagem Estática

Neste trecho, antes de tudos nós inicializamos o MediaPipe e carregamos a imagem estática que será processada.

In [ ]:
# Inicializando MediaPipe
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Define o caminho da imagem a ser processada
img_path = "/content/fiap-ml-visao-computacional/aula-4-analise-facial/imagens/mulher.jpg"

# Lê a imagem do arquivo
image = cv2.imread(img_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Especifica os parâmetros para desenhar os landmarks
drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1)

# Inicializa o modelo FaceMesh
with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.2) as face_mesh:

    # Converte a imagem de BGR para RGB
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Processa a imagem e detecta landmarks
    results = face_mesh.process(rgb_image)

    # Cria uma cópia da imagem para anotação
    annotated_image = image.copy()

    # Itera sobre os landmarks detectados e desenha na imagem anotada
    for face_landmarks in results.multi_face_landmarks:
        mp_drawing.draw_landmarks(
            image=annotated_image,
            landmark_list=face_landmarks,
            connections=mp_face_mesh.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
        )
        mp_drawing.draw_landmarks(
            image=annotated_image,
            landmark_list=face_landmarks,
            connections=mp_face_mesh.FACEMESH_CONTOURS,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style()
        )
        mp_drawing.draw_landmarks(
            image=annotated_image,
            landmark_list=face_landmarks,
            connections=mp_face_mesh.FACEMESH_IRISES,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_iris_connections_style()
        )

# Exibe a imagem anotada
plt.figure(figsize=(20, 10))
plt.subplot(1, 2, 1)
plt.imshow(image)

plt.subplot(1, 2, 2)
plt.imshow(annotated_image)


### 6.2 Detecção de Landmarks em Vídeo


A detecção de *landmarks* faciais é uma etapa crucial em diversas metodologias e algoritmos de análise facial em visão computacional. Aplicações como reconhecimento de expressão facial, estimativa de pose da cabeça e sistemas de detecção de sonolência dependem significativamente das informações de forma facial fornecidas pela detecção desses pontos chave.

O principal objetivo de um algoritmo de detecção de *landmarks* faciais é identificar automaticamente as localizações desses pontos chave em imagens ou vídeos, permitindo uma análise mais aprofundada das características faciais. Os pontos chave, ou *landmarks*, podem ser descritos como pontos dominantes que representam a localização única de um componente facial, como os cantos da boca ou dos olhos. Alternativamente, podem ser pontos interpolados que conectam esses pontos dominantes ao redor dos componentes faciais e do contorno facial, proporcionando uma representação mais completa da face.

<br>
<center><img src="https://developers.google.com/static/mediapipe/images/solutions/examples/face_landmark.png" width=500></center>
<br>

Formalmente, dado um retrato facial denotado como I, um algoritmo de detecção identifica a localização de D *landmarks* representados por suas coordenadas na imagem, $x = {x1, y1, x2, y2, ..., xD, yD}$, onde $x$ e $y$ indicam as coordenadas de imagem dos *landmarks* faciais. Utilizando o *MediaPipe*, podemos realizar a detecção de maneira eficiente e integrada ao processo de análise de pose corporal. A ferramenta oferece um framework robusto para essa tarefa, permitindo não só identificar, mas também acompanhar os *landmarks* faciais em tempo real, proporcionando assim uma solução abrangente para a análise de movimentos e expressões humanas.

Agora vamos usar `import mediapipe as mp` para importar a biblioteca e detectar os pontos de interesse na prática.


In [ ]:
# Inicializando MediaPipe
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Substitua 'path/to/your/video.mp4' pelo caminho do seu vídeo original
file_name = '/content/fiap-ml-visao-computacional/aula-4-analise-facial/videos/jn.mp4'
video_cap = cv2.VideoCapture(file_name)

clip = VideoFileClip('/content/fiap-ml-visao-computacional/aula-4-analise-facial/videos/jn.mp4')
clip.ipython_display(width=500)

#### 6.2.1 Propriedades do Vídeo e Configuração de Saída

Nesta etapa, são extraídas informações referentes às propriedades do vídeo de entrada, incluindo a taxa de quadros por segundo (fps) e as dimensões espaciais dos quadros, representadas por sua largura e altura. O objetivo é assegurar que o arquivo de saída, destinado a armazenar os frames processados com a identificação dos landmarks faciais, mantenha as mesmas características do vídeo original.

In [ ]:
# Pegando propriedades do vídeo original
fps = int(video_cap.get(cv2.CAP_PROP_FPS))
frame_w = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_size = (frame_w, frame_h)

# Configurando o vídeo de saída
video_out_file = file_name[:-4] + '_out_landmarks.mp4'
video_output = cv2.VideoWriter(video_out_file, cv2.VideoWriter_fourcc(*'mp4v'), fps, frame_size)

#### 6.2.2 Detecção e Anotação de Landmarks Faciais

Nesta seção, procede-se com a execução das etapas de detecção e anotação de landmarks faciais nos quadros do vídeo de entrada. Para tal, é instanciado o modelo FaceMesh da biblioteca MediaPipe, configurado para detectar até um rosto por quadro, refinar os landmarks, e operar com um grau de confiança mínimo de 0.5 tanto para detecção quanto para rastreamento.

Inicialmente, cada quadro é lido sequencialmente. O modelo FaceMesh é aplicado sobre a versão RGB do quadro, e os landmarks faciais identificados são sobrepostos à imagem original. Os quadros processados, contendo os landmarks desenhados, são então gravados sequencialmente no arquivo de vídeo de saída.

A seguir, será executada a detecção dos landmarks faciais, a anotação gráfica destes sobre os quadros correspondentes e a gravação destes quadros no arquivo de saída.

In [ ]:
# Configurando o Face Mesh
with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as face_mesh:

  while video_cap.isOpened():
    success, image = video_cap.read()

    if not success:
      print("Finalizando por ter chegado ao final do vídeo.")
      break

    # Processando o frame
    image.flags.writeable = False
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(image_rgb)
    image.flags.writeable = True

    if results.multi_face_landmarks:
      for face_landmarks in results.multi_face_landmarks:
        # Desenhando landmarks no frame
        mp_drawing.draw_landmarks(
            image=image,
            landmark_list=face_landmarks,
            connections=mp_face_mesh.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
        )
        mp_drawing.draw_landmarks(
            image=image,
            landmark_list=face_landmarks,
            connections=mp_face_mesh.FACEMESH_CONTOURS,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles
            .get_default_face_mesh_contours_style())
        mp_drawing.draw_landmarks(
            image=image,
            landmark_list=face_landmarks,
            connections=mp_face_mesh.FACEMESH_IRISES,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles
            .get_default_face_mesh_iris_connections_style())

    # Salvando o frame processado no vídeo de saída
    video_output.write(image)

# Liberando os recursos
video_cap.release()
video_output.release()

print("Vídeo processado e salvo com sucesso:", video_out_file)

In [ ]:
clip = VideoFileClip('/content/fiap-ml-visao-computacional/aula-4-analise-facial/videos/jn_out_landmarks.mp4')
clip.ipython_display(width=500)

## 7. Face Swap

### O que é Face Swap?

**Face Swap** é uma técnica de **processamento de imagens** e **visão computacional** que consiste em **substituir o rosto de uma pessoa pelo de outra em fotos ou vídeos**, preservando expressões, iluminação e perspectiva da cena original.  
Com a evolução da **Inteligência Artificial**, especialmente com **Deep Learning**, essa técnica passou de simples montagens digitais para **deepfakes hiper-realistas**.


## Como funciona tecnicamente?
O processo pode variar conforme a abordagem, mas segue etapas comuns:

1. **Detecção e extração de faces**  
   - Algoritmos detectam o rosto na imagem/vídeo.  
   - Exemplos: `dlib`, `MTCNN`, `MediaPipe`.  

2. **Alinhamento (warping)**  
   - O rosto de origem é ajustado em escala, ângulo e posição para coincidir com o rosto de destino.  

3. **Mapeamento e substituição**  
   - Textura do rosto de origem é mapeada sobre o de destino.  
   - Abordagens:
     - **Tradicional (OpenCV)**: morphing geométrico simples.  
     - **IA (Deepfake)**: uso de **Autoencoders** ou **GANs (Generative Adversarial Networks)** para gerar rostos realistas.  

4. **Pós-processamento**  
   - Correção de cor, iluminação e bordas para tornar a substituição imperceptível.  



## Histórico
- **2010–2015**: primeiros experimentos acadêmicos com *morphing facial*.  
- **2017**: surge o termo **deepfake**, popularizado em fóruns online, combinando "deep learning" + "fake".  
- **2018 em diante**: democratização com ferramentas open-source (ex.: *DeepFaceLab*, *Faceswap*).  
- **Atualmente**: uso em **cinema, publicidade, games** e também em **fraudes digitais**.



## Aplicações legítimas
- **Entretenimento e cinema**: rejuvenescimento de atores, dublagem e efeitos especiais.  
- **Avatares digitais e realidade aumentada**.  
- **Privacidade em datasets**: substituição de rostos em pesquisas médicas e educacionais.  



## Riscos e controvérsias
- **Desinformação**: criação de fake news e manipulação política.  
- **Fraudes digitais**: golpes usando identidade falsa.  
- **Questões éticas e legais**: direito de imagem, consentimento e privacidade.  



## Ética e regulação
- **Consentimento explícito** deve ser a regra.  
- **Uso acadêmico**: permitido quando há anonimização ou substituição para preservar privacidade.  
- **Legislação**: vários países já discutem leis específicas contra deepfakes maliciosos.  


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def bgr2rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def imshow(ax, img, title=None, cmap=None):
    ax.imshow(img if cmap is None else img, cmap=cmap)
    ax.axis("off")
    if title:
        ax.set_title(title)


def load_detectors():
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_alt2.xml")
    eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")
    if face_cascade.empty() or eye_cascade.empty():
        raise RuntimeError("Falha ao carregar Haarcascades. Verifique a instalação do OpenCV.")
    return face_cascade, eye_cascade

def detect_largest_face(gray, face_cascade):
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))
    if len(faces) == 0:
        return None
    # Escolhe o maior rosto (mais provável ser o principal)
    faces = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)
    return faces[0]  # (x, y, w, h)

def detect_eyes_in_roi(gray_roi, eye_cascade):
    # Detecta olhos na ROI; retornamos pelo menos dois, priorizando maiores
    eyes = eye_cascade.detectMultiScale(gray_roi, scaleFactor=1.1, minNeighbors=5, minSize=(20, 20))
    if len(eyes) < 2:
        return None
    eyes = sorted(eyes, key=lambda e: e[2]*e[3], reverse=True)[:2]
    return eyes


def align_by_eyes(image, face_rect, eye_boxes):
    (x, y, w, h) = face_rect
    # Coordenadas absolutas dos olhos (centros)
    e1 = (x + eye_boxes[0][0] + eye_boxes[0][2]//2, y + eye_boxes[0][1] + eye_boxes[0][3]//2)
    e2 = (x + eye_boxes[1][0] + eye_boxes[1][2]//2, y + eye_boxes[1][1] + eye_boxes[1][3]//2)

    # Garante ordem: olho esquerdo tem menor x
    left_eye, right_eye = (e1, e2) if e1[0] < e2[0] else (e2, e1)

    # Ângulo da linha dos olhos
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    angle = np.degrees(np.arctan2(dy, dx))

    # Rotação em torno do centro do rosto
    center = (int(x + w // 2), int(y + h // 2))
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    aligned = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]), flags=cv2.INTER_LINEAR)

    return aligned, (left_eye, right_eye), angle, M


def elliptical_face_mask(size, feather=0.15):
    h, w = size
    mask = np.zeros((h, w), dtype=np.uint8)

    center = (w // 2, h // 2)
    axes   = (int(w * 0.45), int(h * 0.60))
    cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)

    # Feathering (desfocagem suave na borda da elipse)
    k = int(min(w, h) * feather)
    if k % 2 == 0:
        k += 1
    if k < 3:
        k = 3
    mask = cv2.GaussianBlur(mask, (k, k), 0)
    return mask


def face_swap_with_plots(source_path, target_path, output_path="swap_result.jpg"):
    face_cascade, eye_cascade = load_detectors()

    src = cv2.imread(source_path)
    dst = cv2.imread(target_path)
    if src is None or dst is None:
        raise FileNotFoundError("Falha ao ler imagens. Verifique os caminhos de source/target.")

    # 0) Originais
    fig0, axs0 = plt.subplots(1, 2, figsize=(10, 5))
    imshow(axs0[0], bgr2rgb(src), "Source (BGR→RGB)")
    imshow(axs0[1], bgr2rgb(dst), "Target (BGR→RGB)")
    plt.show()

    # 1) Detecção de rostos
    src_gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
    dst_gray = cv2.cvtColor(dst, cv2.COLOR_BGR2GRAY)

    src_face = detect_largest_face(src_gray, face_cascade)
    dst_face = detect_largest_face(dst_gray, face_cascade)
    if src_face is None or dst_face is None:
        raise RuntimeError("Não foi possível detectar rosto em uma das imagens.")

    src_boxes = src.copy()
    dst_boxes = dst.copy()
    sx, sy, sw, sh = src_face
    dx, dy, dw, dh = dst_face
    cv2.rectangle(src_boxes, (sx, sy), (sx+sw, sy+sh), (0, 255, 255), 2)
    cv2.rectangle(dst_boxes, (dx, dy), (dx+dw, dy+dh), (0, 255, 0), 2)

    fig1, axs1 = plt.subplots(1, 2, figsize=(10, 5))
    imshow(axs1[0], bgr2rgb(src_boxes), "Source: Face bbox")
    imshow(axs1[1], bgr2rgb(dst_boxes), "Target: Face bbox")
    plt.show()

    # 2) Detecção de olhos + alinhamento (source)
    src_roi_gray = src_gray[sy:sy+sh, sx:sx+sw]
    src_eyes = detect_eyes_in_roi(src_roi_gray, eye_cascade)

    eyes_vis = src.copy()
    angle = 0.0
    if src_eyes is not None and len(src_eyes) >= 2:
        # desenha olhos relativos à ROI na imagem original
        for (ex, ey, ew, eh) in src_eyes:
            cv2.rectangle(eyes_vis, (sx+ex, sy+ey), (sx+ex+ew, sy+ey+eh), (255, 0, 0), 2)

        aligned_src, (left_eye, right_eye), angle, M = align_by_eyes(src, src_face, src_eyes)
        aligned_src_gray = cv2.cvtColor(aligned_src, cv2.COLOR_BGR2GRAY)
        # Redetecta o rosto após alinhamento
        src_face_aligned = detect_largest_face(aligned_src_gray, face_cascade)
        if src_face_aligned is None:
            raise RuntimeError("Falha ao redetectar rosto após alinhamento.")
        sx, sy, sw, sh = src_face_aligned
    else:
        # Sem olhos detectados: segue com a imagem original
        aligned_src = src.copy()

    # Visual da etapa de olhos/alinhamento
    fig2, axs2 = plt.subplots(1, 3, figsize=(15, 5))
    imshow(axs2[0], bgr2rgb(src), "Source (bruta)")
    imshow(axs2[1], bgr2rgb(eyes_vis), "Source: olhos detectados")
    title_aligned = f"Source alinhada (ângulo ≈ {angle:.2f}°)" if src_eyes is not None else "Source (sem alinhamento)"
    # Desenha a bbox após alinhamento
    aligned_vis = aligned_src.copy()
    cv2.rectangle(aligned_vis, (sx, sy), (sx+sw, sy+sh), (0, 255, 255), 2)
    imshow(axs2[2], bgr2rgb(aligned_vis), title_aligned)
    plt.show()

    # 3) Extração (crop) e warping (resize para a bbox de destino)
    src_face_crop = aligned_src[sy:sy+sh, sx:sx+sw]
    if src_face_crop.size == 0:
        raise RuntimeError("Recorte do rosto origem vazio.")

    src_face_resized = cv2.resize(src_face_crop, (dw, dh), interpolation=cv2.INTER_LINEAR)

    fig3, axs3 = plt.subplots(1, 3, figsize=(15, 5))
    imshow(axs3[0], bgr2rgb(aligned_src), "Source (após alinhamento)")
    imshow(axs3[1], bgr2rgb(src_face_crop), "Crop da face (source)")
    imshow(axs3[2], bgr2rgb(src_face_resized), "Face redimensionada → bbox destino")
    plt.show()

    # 4) Máscara elíptica
    mask = elliptical_face_mask((dh, dw))
    mask_rgb = np.dstack([mask, mask, mask])

    fig4, axs4 = plt.subplots(1, 2, figsize=(10, 5))
    imshow(axs4[0], mask, "Máscara elíptica", cmap="gray")
    temp_preview = dst.copy()
    temp_preview[dy:dy+dh, dx:dx+dw] = np.where(mask_rgb > 0, src_face_resized, temp_preview[dy:dy+dh, dx:dx+dw])
    imshow(axs4[1], bgr2rgb(temp_preview), "Preview: face inserida + máscara (sem blend Poisson)")
    plt.show()

    # 5) Blend Poisson (seamlessClone)
    center = (dx + dw // 2, dy + dh // 2)
    mixed = cv2.seamlessClone(src_face_resized, dst, mask, center, cv2.MIXED_CLONE)

    fig5, axs5 = plt.subplots(1, 2, figsize=(10, 5))
    imshow(axs5[0], bgr2rgb(dst), "Target (original)")
    imshow(axs5[1], bgr2rgb(mixed), "Resultado final (seamlessClone)")
    plt.show()

    # Persistência do resultado
    cv2.imwrite(output_path, mixed)
    return output_path


face_swap_with_plots("/content/fiap-ml-visao-computacional/aula-4-analise-facial/imagens/pessoa1.jpg", "/content/fiap-ml-visao-computacional/aula-4-analise-facial/imagens/pessoa2.jpg", "swap_result.jpg")
